In [1]:
import pandas as pd
import numpy as np

In [25]:
df=pd.read_csv('all_kindle_review.csv')
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [26]:
df=df[['reviewText','rating']]
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [27]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [28]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [29]:
df['rating']=df['rating'].apply(lambda x:0 if x<=3 else 1)
df['rating'].unique()

array([0, 1])

In [30]:
df['rating'].value_counts()

rating
0    6000
1    6000
Name: count, dtype: int64

In [31]:
df['reviewText']=df['reviewText'].str.lower()

In [33]:
import nltk
import re
from nltk.corpus import stopwords
from bs4 import BeautifulSoup


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
stop_words=stopwords.words('english')

In [39]:
#remove html tags
df['reviewText']=df['reviewText'].apply(lambda x: BeautifulSoup(x,'html.parser').get_text())
#remove url
df['reviewText']=df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
#remove special characters
df['reviewText']=df['reviewText'].apply(lambda x: re.sub('[^a-z A-Z 0-9-]+','',x))
#remove stop words
df['reviewText']=df['reviewText'].apply(lambda x: " ".join([words for words in x.split() if x not in stop_words]))
#remove extra spaces
df['reviewText']=df['reviewText'].apply(lambda x:" ".join(x.split()))


In [40]:
from nltk.stem import WordNetLemmatizer
lemmatizer=WordNetLemmatizer()

In [41]:
def lemmatizer_text(Text):
    text=[lemmatizer.lemmatize(word) for word in Text.split()]
    return " ".join(text)

In [42]:
df['reviewText']=df['reviewText'].apply(lambda x:lemmatizer_text(x))

In [43]:
df.head()

,reviewText,rating
0,jace rankin may be short but he nothing to mes...,0
1,great short read i didnt want to put it down s...,1
2,ill start by saying this is the first of four ...,0
3,aggie is angela lansbury who carry pocketbook ...,0
4,i did not expect this type of book to be in li...,1


In [45]:
#train test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['reviewText'],df['rating'],test_size=0.2,random_state=42)


In [47]:
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
X_train_bow=bow.fit_transform(X_train)
X_test_bow=bow.transform(X_test)

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
X_train_tfidf=tfidf.fit_transform(X_train)
X_test_tfidf=tfidf.transform(X_test)


In [49]:
from sklearn.naive_bayes import GaussianNB
gnb=GaussianNB()

In [52]:
#BOW
gnb.fit(X_train_bow.toarray(),y_train)
y_pred_bow=gnb.predict(X_test_bow.toarray())

In [53]:
#tfidf
gnb.fit(X_train_tfidf.toarray(),y_train)
y_pred_tfidf=gnb.predict(X_test_tfidf.toarray())


In [54]:
#accuracy
from sklearn.metrics import accuracy_score
print(f'accuracy with bow {accuracy_score(y_pred_bow,y_test)}')
print(f'accuracy with tfidf {accuracy_score(y_pred_tfidf,y_test)}')

accuracy with bow 0.58875
accuracy with tfidf 0.6204166666666666
